# Storybound Library
### Find the book that feels like the story you have in mind.

Storybound is a library discovery concept: describe a plot, mood, character, or setting and get a ranked shelf of books that may fit. Explore a small sample catalog, narrow by genre, and send a demo request for a title. The interface uses a quiet paper-like background, crisp high contrast cards, and a custom cursor with gently following book and pencil accents.

> **Demo note:** the catalog and availability are sample data. The request button demonstrates a library request flow; it does not contact a real library or reserve a physical book.

## 1. Install the libraries
Run this cell in a notebook environment with internet access. The embedding model downloads the first time it is loaded.

In [ ]:
!pip -q install gradio sentence-transformers scikit-learn

## 2. Create the sample catalog
Each book has a short synopsis and story tags. Add or replace entries in `books` to tailor the demo to your collection.

In [ ]:
books = [
    {"title":"The Hobbit", "author":"J. R. R. Tolkien", "genre":"Fantasy", "year":1937, "availability":"Available", "synopsis":"A comfort-loving hobbit joins a band of dwarves on a perilous quest to reclaim their mountain home from a dragon. A warm adventure of courage, friendship, riddles, and a surprisingly brave hero.", "tags":"cozy adventure unlikely hero friendship dragons journey courage folklore"},
    {"title":"The Night Circus", "author":"Erin Morgenstern", "genre":"Fantasy", "year":2011, "availability":"Available", "synopsis":"Two young magicians are bound to a mysterious competition staged inside an enchanting circus that appears without warning. Their rivalry turns into a tender, atmospheric story of wonder, secrets, and impossible choices.", "tags":"magical circus romance mysterious atmospheric wonder rivalry dreamlike"},
    {"title":"A Psalm for the Wild-Built", "author":"Becky Chambers", "genre":"Science Fiction", "year":2021, "availability":"Available", "synopsis":"On a gentle moon where robots gained freedom long ago, a tea monk meets a curious robot in search of what people need. A hopeful, contemplative journey about purpose, rest, and belonging.", "tags":"hopeful gentle robot nature purpose healing cozy philosophical future"},
    {"title":"The Martian", "author":"Andy Weir", "genre":"Science Fiction", "year":2011, "availability":"Checked out", "synopsis":"Stranded alone on Mars, an astronaut uses engineering, humor, and stubborn optimism to survive while people on Earth race to bring him home. A resourceful, high-stakes survival adventure.", "tags":"space survival science humor isolated problem solving rescue suspense"},
    {"title":"Pride and Prejudice", "author":"Jane Austen", "genre":"Classics", "year":1813, "availability":"Available", "synopsis":"Elizabeth Bennet navigates family expectations, social manners, and her sharp first impressions of the proud Mr Darcy. A witty romance about self-knowledge, class, and learning to see people clearly.", "tags":"witty romance family society manners misunderstandings slow burn"},
    {"title":"The Thursday Murder Club", "author":"Richard Osman", "genre":"Mystery", "year":2020, "availability":"Available", "synopsis":"Four retirement-village friends meet to investigate old cases, then find themselves close to a real murder. A charming, funny mystery powered by friendship, clever clues, and unexpected warmth.", "tags":"cozy mystery funny friendship amateur detectives village clues"},
    {"title":"The House in the Cerulean Sea", "author":"TJ Klune", "genre":"Fantasy", "year":2020, "availability":"Available", "synopsis":"A by-the-book case worker visits a remote home for magical children and discovers a family that changes his understanding of care and belonging. A whimsical, kind-hearted story about found family.", "tags":"found family kindness magical children whimsical heartwarming belonging"},
    {"title":"And Then There Were None", "author":"Agatha Christie", "genre":"Mystery", "year":1939, "availability":"Available", "synopsis":"Ten strangers arrive on an isolated island and are accused of hidden crimes. As guests disappear one by one, suspicion and tension build in this tightly plotted locked-room classic.", "tags":"suspenseful isolated island secrets classic puzzle tense whodunit"},
    {"title":"The Book Thief", "author":"Markus Zusak", "genre":"Historical Fiction", "year":2005, "availability":"Available", "synopsis":"A young girl in wartime Germany finds refuge in stolen books and the kindness of her foster family. Narrated by Death, this moving story explores words, courage, grief, and small acts of resistance.", "tags":"historical moving books family courage war compassion resilient"},
    {"title":"The Very Secret Society of Irregular Witches", "author":"Sangu Mandanna", "genre":"Fantasy", "year":2022, "availability":"Available", "synopsis":"A lonely witch is invited to teach three young witches at an isolated house, where an unexpected household becomes a home. A light, magical romance about found family and letting people in.", "tags":"witches cozy romance found family warm funny magical home"},
    {"title":"The Ocean at the End of the Lane", "author":"Neil Gaiman", "genre":"Fantasy", "year":2013, "availability":"Checked out", "synopsis":"A man revisits his childhood home and remembers a strange, haunting adventure involving a mysterious girl and forces beyond ordinary life. A lyrical, eerie tale about memory, fear, and childhood.", "tags":"eerie childhood memory lyrical magical dark folklore haunting"},
    {"title":"Before the Coffee Gets Cold", "author":"Toshikazu Kawaguchi", "genre":"Literary Fiction", "year":2015, "availability":"Available", "synopsis":"In a Tokyo café, visitors can travel to the past under strict rules, including returning before their coffee grows cold. Four intimate stories explore regret, love, family, and the present.", "tags":"reflective time travel café family love regret gentle emotional"},
    {"title":"The Hunger Games", "author":"Suzanne Collins", "genre":"Young Adult", "year":2008, "availability":"Available", "synopsis":"In a divided future, Katniss volunteers to take her sister's place in a televised fight for survival. A tense story of defiance, sacrifice, media, and the bonds that keep people human.", "tags":"dystopian survival rebellion sisterhood action courage young adult"},
    {"title":"The Lightning Thief", "author":"Rick Riordan", "genre":"Young Adult", "year":2005, "availability":"Available", "synopsis":"A modern boy learns he is connected to Greek mythology and sets out on a cross-country quest with new friends. Fast, funny, and full of monsters, friendship, and self-discovery.", "tags":"funny mythology quest friendship monsters modern adventure young adult"},
    {"title":"The Seven Husbands of Evelyn Hugo", "author":"Taylor Jenkins Reid", "genre":"Historical Fiction", "year":2017, "availability":"Available", "synopsis":"An aging Hollywood icon tells the complicated story of her rise, her seven marriages, and the great love she kept private. A glamorous, emotional portrait of ambition, identity, and secrets.", "tags":"glamorous Hollywood secrets romance identity ambition emotional"},
    {"title":"The Silent Patient", "author":"Alex Michaelides", "genre":"Thriller", "year":2019, "availability":"Available", "synopsis":"A psychotherapist becomes obsessed with learning why a celebrated artist stopped speaking after a shocking act of violence. A twist-driven psychological mystery about obsession and hidden motives.", "tags":"psychological suspense twist secrets obsession dark mystery"},
    {"title":"Legends & Lattes", "author":"Travis Baldree", "genre":"Fantasy", "year":2022, "availability":"Available", "synopsis":"A retired orc adventurer opens a coffee shop and builds a new life with the help of an unusual found family. A low-stakes fantasy about fresh starts, friendship, and following a small dream.", "tags":"cozy low stakes coffee shop friendship fresh start found family fantasy"},
    {"title":"Educated", "author":"Tara Westover", "genre":"Memoir", "year":2018, "availability":"Available", "synopsis":"Raised in a strict, isolated family, a young woman pursues education and builds a life beyond the world she knew. A memoir about resilience, family, memory, and the power of learning.", "tags":"memoir resilience education family identity personal growth"}
]

## 3. Build story-based matching
The app embeds each synopsis and its tags, then compares the reader’s description with the catalog using cosine similarity. Genre and availability preferences shape the ranked shelf.

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import gradio as gr

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
book_texts = [f"{b['title']} by {b['author']}. {b['genre']}. {b['synopsis']} Story themes: {b['tags']}." for b in books]
book_vectors = model.encode(book_texts, normalize_embeddings=True, show_progress_bar=False)
print(f"Storybound is ready with {len(books)} sample books.")

## 4. Create recommendation and request handlers

In [ ]:
def find_books(story, genre, available_only, count):
    if not story or not story.strip():
        return "Tell us a little about the story you want to read, and we’ll find a shelf for you.", gr.update(choices=[], value=None)
    query_vector = model.encode([story.strip()], normalize_embeddings=True)
    scores = cosine_similarity(query_vector, book_vectors)[0]
    ranked = sorted(enumerate(scores), key=lambda item: item[1], reverse=True)
    cards = []
    choices = []
    shown = 0
    for idx, score in ranked:
        book = books[idx]
        if genre != "Any genre" and book["genre"] != genre:
            continue
        if available_only and book["availability"] != "Available":
            continue
        shown += 1
        choices.append(book["title"])
        status = "● Available" if book["availability"] == "Available" else "○ Checked out"
        cards.append(
            f"### {shown}. {book['title']}  \n"
            f"**{book['author']}** · {book['genre']} · {book['year']} · {status}  \n"
            f"{book['synopsis']}  \n"
            f"*Story match: {score:.0%}*"
        )
        if shown >= int(count):
            break
    if not cards:
        return "No matching books in this sample shelf. Try another genre or turn off ‘Available now’.", gr.update(choices=[], value=None)
    return "\n\n---\n\n".join(cards), gr.update(choices=choices, value=choices[0] if choices else None)

def request_book(title):
    if not title:
        return "Choose a book from your story shelf first."
    return f"Request noted for **{title}**. This is a demo confirmation; connect a library catalog to place a real hold."


## 5. Design the calm, content-first interface
The background stays visually quiet while the cards carry the contrast and readable typography. A custom cursor and a few low-contrast book/pencil accents follow mouse movement with gentle lag.

In [ ]:
css = r"""
@import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@400;500;600;700&family=Playfair+Display:wght@600;700&display=swap');
:root { --ink:#23352f; --muted:#64736c; --paper:#f5f4ee; --card:#fffefa; --line:#e4e5dc; --green:#315d4b; --green2:#234637; --gold:#c49a53; }
.gradio-container { position:relative;z-index:1;background:transparent!important;color:var(--ink)!important;font-family:'DM Sans',sans-serif!important;max-width:1180px!important; }
body { background:var(--paper)!important; }
body { cursor:none!important; }
#cursor-dot { position:fixed; z-index:99999; left:0;top:0; width:13px;height:13px; border:2px solid var(--green); border-radius:50%; pointer-events:none; transform:translate(-50%,-50%); background:#fffefa; transition:width .16s,height .16s,background .16s; }
#cursor-dot.active { width:23px;height:23px;background:#dfe9dfaa; }
#floaties { position:fixed;inset:0;pointer-events:none;z-index:0;overflow:hidden; }
.floaty { position:absolute;left:0;top:0;will-change:transform;opacity:.20;filter:saturate(.68);font-size:25px;user-select:none; }
#hero { padding:34px 36px 30px; border:1px solid var(--line); border-radius:22px; background:var(--card); box-shadow:0 8px 28px #24382d0a; margin:10px 0 20px; }
#hero h1 { font:700 clamp(2.2rem,5vw,3.4rem)/1.05 'Playfair Display',serif; letter-spacing:-.04em; color:var(--ink); margin:7px 0 10px; }
#hero p { color:var(--muted);font-size:1.05rem;max-width:680px;line-height:1.7; }
.eyebrow { color:var(--green);font-size:.76rem;font-weight:700;letter-spacing:.14em;text-transform:uppercase; }
.panel { background:var(--card);border:1px solid var(--line);border-radius:18px;padding:20px; }
label, .label-wrap span { color:var(--ink)!important;font-weight:600!important; }
input,textarea,.wrap, .block { border-color:var(--line)!important; }
textarea, input { background:#fff!important;color:var(--ink)!important; }
button.primary { background:var(--green)!important;border-color:var(--green)!important;color:white!important; }
button.primary:hover { background:var(--green2)!important; }
#results { background:var(--card);border:1px solid var(--line);border-radius:18px;padding:22px 26px;line-height:1.75;color:var(--ink); }
#results h3 { font:700 1.3rem 'Playfair Display',serif;color:var(--ink); }
#results p { color:#46574f; }
#results hr { border-color:var(--line); }
#small-note { color:var(--muted);font-size:.86rem; }
@media (pointer:coarse) { body { cursor:auto!important; } #cursor-dot,#floaties { display:none; } }
"""
js = r"""
() => {
  if (window.__storyboundCursorReady) return;
  window.__storyboundCursorReady = true;
  const dot = document.createElement('div'); dot.id='cursor-dot'; document.body.appendChild(dot);
  const layer = document.createElement('div'); layer.id='floaties'; document.body.appendChild(layer);
  const icons = ['📚','📖','✏️','📕','🖋️','📗','📘','✒️','📙','📚','✏️','📖','📔','🖊️','📕','📓','✏️','📚','📖','📗','🖋️','📘'];
  const sprites = icons.map((icon,i) => {
    const e=document.createElement('span'); e.className='floaty'; e.textContent=icon; layer.appendChild(e);
    return {e, x:Math.random()*innerWidth, y:Math.random()*innerHeight,
      vx:(Math.random()-.5)*.32, vy:(Math.random()-.5)*.32,
      phase:Math.random()*Math.PI*2, drift:.18+Math.random()*.25, radius:105+Math.random()*28};
  });
  let mx=-1000,my=-1000,frame=0;
  window.addEventListener('pointermove', ev => {
    mx=ev.clientX; my=ev.clientY;
    dot.style.transform=`translate(${mx}px,${my}px) translate(-50%,-50%)`;
  }, {passive:true});
  window.addEventListener('pointerdown',()=>dot.classList.add('active'));
  window.addEventListener('pointerup',()=>dot.classList.remove('active'));
  function animate(){
    frame += .012;
    for(const s of sprites){
      // Slow ambient drift keeps the icons moving even when the pointer is still.
      s.vx += Math.sin(frame+s.phase)*.0012;
      s.vy += Math.cos(frame*1.13+s.phase)*.0012;
      const dx=s.x-mx, dy=s.y-my, distance=Math.hypot(dx,dy);
      // A soft outward push makes each icon dodge the pointer, then drift back naturally.
      if(distance<s.radius && distance>0){
        const force=(1-distance/s.radius)*.075;
        s.vx += (dx/distance)*force; s.vy += (dy/distance)*force;
      }
      s.vx*=.992; s.vy*=.992;
      s.x+=s.vx; s.y+=s.vy;
      if(s.x < -35) s.x=innerWidth+25; if(s.x>innerWidth+35) s.x=-25;
      if(s.y < -35) s.y=innerHeight+25; if(s.y>innerHeight+35) s.y=-25;
      s.e.style.transform=`translate3d(${s.x}px,${s.y}px,0) rotate(${s.vx*24}deg)`;
    }
    requestAnimationFrame(animate);
  }
  animate();
}
"""

with gr.Blocks(css=css, js=js, theme=gr.themes.Base(primary_hue="green", neutral_hue="stone"), title="Storybound Library") as demo:
    gr.HTML("""<section id='hero'><div class='eyebrow'>STORYBOUND · A LIBRARY DISCOVERY DESK</div><h1>Find your next<br>world to get lost in.</h1><p>Describe the story you’re craving — a setting, a feeling, a character, or a twist — and we’ll pull a few books from the shelf for you.</p></section>""")
    with gr.Row(equal_height=True):
        with gr.Column(scale=5, elem_classes="panel"):
            gr.Markdown("### What kind of story are you looking for?")
            story = gr.Textbox(label="Your story", lines=5, placeholder="A quiet, hopeful story about a tired traveler who finds a new home among kind strangers…")
            gr.Examples(
                examples=[
                    ["A cozy mystery where a group of clever friends solve a murder in a quiet village."],
                    ["A hopeful story about a lonely person who finds a kind found family in a magical house."],
                    ["A funny space survival adventure with science, an isolated hero, and a daring rescue."],
                    ["A gentle, reflective tale about time travel, family, and a second chance."],
                    ["A brave young hero goes on a funny mythology-filled quest with new friends."],
                    ["A tense locked-room mystery on a remote island, full of secrets and twists."],
                ],
                inputs=story,
                label="Try a sample story prompt",
                examples_per_page=3,
            )
            with gr.Row():
                genre = gr.Dropdown(["Any genre"] + sorted({b["genre"] for b in books}), value="Any genre", label="Genre")
                count = gr.Slider(3, 8, value=5, step=1, label="Books on your shelf")
            available = gr.Checkbox(label="Available now", value=False)
            discover = gr.Button("Find my story shelf", variant="primary", size="lg")
            gr.Markdown("*Descriptions are matched by meaning and story themes, not just exact keywords.*", elem_id="small-note")
        with gr.Column(scale=7):
            gr.Markdown("### Your story shelf")
            results = gr.Markdown("Your recommendations will appear here — tell us the story you have in mind.", elem_id="results")
            gr.Markdown("### Request a book")
            with gr.Row():
                selected = gr.Dropdown(choices=[], label="Choose from your recommendations", scale=3)
                request = gr.Button("Request this book", variant="primary", scale=1)
            confirmation = gr.Markdown()
    discover.click(find_books, inputs=[story,genre,available,count], outputs=[results,selected])
    story.submit(find_books, inputs=[story,genre,available,count], outputs=[results,selected])
    available.change(find_books, inputs=[story,genre,available,count], outputs=[results,selected])
    genre.change(find_books, inputs=[story,genre,available,count], outputs=[results,selected])
    count.change(find_books, inputs=[story,genre,available,count], outputs=[results,selected])
    request.click(request_book, inputs=selected, outputs=confirmation)
    gr.Markdown("Storybound demo · Sample catalog only · Requests are not sent to a real library", elem_id="small-note")

demo.launch()

## 6. Launch the library
Run the final cell and open the local Gradio link. The first run may take a little longer while the language embedding model downloads.